# Intrusion Detection on CICIDS2017 — Reproduction & Critique

**Course:** Data Science Methods in Cyber Security (Dr. Uri Itai, University of Haifa)  
**Student:** Eyal Steinmetz  
**Deadline:** Friday, 10 July 2026, 23:59

**Mission:** Reproduce Rodríguez et al. (2022, *Sensors* MDPI), who claim a Random Forest reaches
**F1 > 0.999** on CICIDS2017, then demonstrate that the claim is an artifact of three documented
methodological flaws — duplicate contamination, temporal leakage, and aggregate-only metrics —
using Engelen et al. (2021) as peer-reviewed counter-evidence.

| Notebook section | Report section | Purpose |
|---|---|---|
| 0. Setup | front matter | Imports, seed, paths, figure helper |
| **1. Data Loading & Cleaning** | §3 Data | Load 8 day-files, tag `day`, strip columns, quantify quality issues |
| 2. EDA | §4 EDA | Class balance, temporal structure, Spearman correlation |
| 3. Feature Engineering | §5 | Encoding, log-transform, correlation pruning |
| 4. Model Training | §6 | Strategy A (random) vs Strategy B (dedup + temporal) |
| 5. Evaluation | §6/§7 | Metric suite, confusion matrices, ROC/PR |
| 6. Error Analysis & Critique | §2 | Three-flaw framework, per-class failure, verdict |
| 7. Executive Summary | §1/§8 | Problem → method → findings → verdict |

> This session builds **Section 0 (Setup)** and **Section 1 (Data Loading & Cleaning)** — Phase 2 in the project plan.

## Section 0 — Setup & Imports

We fix a single global random seed (`RANDOM_STATE = 42`) used by every split, model, and SMOTE
call for reproducibility, define the project paths, and provide a `save_fig()` helper so every
figure lands in `figures/` at a consistent resolution. The notebook lives in `notebooks/`, so
all paths are relative to that directory.

In [1]:
# Standard library
import warnings
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility — one seed used everywhere
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paths (notebook runs from notebooks/)
DATA_DIR = Path("../data/MachineLearningCVE")
FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(exist_ok=True)

# Warnings policy: silence only FutureWarning noise from pandas/sklearn version churn.
# We do NOT silence SettingWithCopyWarning — that one signals real bugs and must surface.
warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_theme(style="whitegrid")


def save_fig(fig, name, tight=True):
    """Save a figure to figures/ with consistent settings, then close it."""
    path = FIGURES_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight" if tight else None)
    plt.close(fig)
    print(f"Saved: {path}")


print("Imports OK")
print(f"RANDOM_STATE = {RANDOM_STATE}")
print(f"DATA_DIR     = {DATA_DIR.resolve()}")
print(f"FIGURES_DIR  = {FIGURES_DIR.resolve()}")

Imports OK
RANDOM_STATE = 42
DATA_DIR     = C:\Users\אייל. ש\Documents\HUP\cyber\project\data\MachineLearningCVE
FIGURES_DIR  = C:\Users\אייל. ש\Documents\HUP\cyber\project\figures


## Section 1 — Data Loading & Cleaning

We load CICIDS2017 from its pre-extracted CSV flow features (`MachineLearningCVE/`). The published
release ships **8** CSV files, not 5: Thursday is split into *WebAttacks* and *Infiltration*
captures, and Friday into *Morning*, *PortScan*, and *DDoS* captures. Because the ML CSVs carry no
timestamp column, we derive the weekday — needed later for the temporal split (Flaw 2) — directly
from each filename.

### 1.1 — Load all day-files and concatenate

Each CSV is one capture. We tag every row with its source weekday (`day_name`) and an ordinal
`day` index (Mon=0 … Fri=4) *before* concatenating, then strip whitespace from column names
immediately — CICFlowMeter emits leading/trailing spaces in headers (Engelen et al., 2021).

In [2]:
# Ordinal weekday encoding for the temporal split (Mon=0 .. Fri=4)
WEEKDAY_ORDER = {"Monday": 0, "Tuesday": 1, "Wednesday": 2, "Thursday": 3, "Friday": 4}


def weekday_from_filename(filename: str):
    """Derive (weekday_name, ordinal) from a CICIDS2017 CSV filename.

    The 8-file release prefixes every file with the weekday, e.g.
    'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv'. Case-insensitive
    because the Wednesday file uses a lowercase 'workingHours'.
    """
    low = filename.lower()
    for name, idx in WEEKDAY_ORDER.items():
        if low.startswith(name.lower()):
            return name, idx
    raise ValueError(f"Cannot derive weekday from filename: {filename}")


csv_paths = sorted(DATA_DIR.glob("*.pcap_ISCX.csv"))
assert csv_paths, f"No CSVs found in {DATA_DIR.resolve()} — place the MachineLearningCVE files there."

frames = []
for path in csv_paths:
    day_name, day_idx = weekday_from_filename(path.name)
    part = pd.read_csv(path, low_memory=False)
    part.columns = part.columns.str.strip()  # strip per-file before concat
    part["day"] = day_idx
    part["day_name"] = day_name
    frames.append(part)
    print(f"  {path.name:<55} -> {day_name:<9} ({len(part):>8,} rows)")

df = pd.concat(frames, ignore_index=True)
df.columns = df.columns.str.strip()  # idempotent guard after concat

print(f"\nLoaded {len(df):,} rows x {len(df.columns)} columns from {len(csv_paths)} files")
print("\nRows per day:")
print(df["day_name"].value_counts().reindex(WEEKDAY_ORDER.keys()).to_string())

  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv        -> Friday    ( 225,745 rows)


  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv    -> Friday    ( 286,467 rows)


  Friday-WorkingHours-Morning.pcap_ISCX.csv               -> Friday    ( 191,033 rows)


  Monday-WorkingHours.pcap_ISCX.csv                       -> Monday    ( 529,918 rows)


  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv -> Thursday  ( 288,602 rows)


  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv  -> Thursday  ( 170,366 rows)


  Tuesday-WorkingHours.pcap_ISCX.csv                      -> Tuesday   ( 445,909 rows)


  Wednesday-workingHours.pcap_ISCX.csv                    -> Wednesday ( 692,703 rows)



Loaded 2,830,743 rows x 81 columns from 8 files

Rows per day:


day_name
Monday       529918
Tuesday      445909
Wednesday    692703
Thursday     458968
Friday       703245


### 1.2 — First inspection: dtypes, columns, labels

We confirm the column names are clean, locate the only expected non-numeric feature columns
(`Label` plus our two tag columns), flag the known redundant `Fwd Header Length.1` duplicate
column, and print the full label distribution. The label counts must show the severe imbalance
(~80% BENIGN) and the ultra-rare attack classes (Heartbleed, SQL Injection, Infiltration) that
later expose Flaw 3.

In [3]:
# Non-numeric columns (expect only Label + our two tag columns)
non_numeric = df.select_dtypes(exclude=np.number).columns.tolist()
print(f"Non-numeric columns ({len(non_numeric)}): {non_numeric}")

# Known redundant column from CICFlowMeter (Engelen et al., 2021) — flag now, drop in Phase 3
dup_col = "Fwd Header Length.1"
print(f"\n'{dup_col}' present (redundant, drop later): {dup_col in df.columns}")

print(f"\nTotal feature columns (excl. Label/day/day_name): {len(df.columns) - 3}")
print("\nAll column names:")
for i, c in enumerate(df.columns):
    print(f"  [{i:>2}] {c}")

Non-numeric columns (2): ['Label', 'day_name']

'Fwd Header Length.1' present (redundant, drop later): True

Total feature columns (excl. Label/day/day_name): 78

All column names:
  [ 0] Destination Port
  [ 1] Flow Duration
  [ 2] Total Fwd Packets
  [ 3] Total Backward Packets
  [ 4] Total Length of Fwd Packets
  [ 5] Total Length of Bwd Packets
  [ 6] Fwd Packet Length Max
  [ 7] Fwd Packet Length Min
  [ 8] Fwd Packet Length Mean
  [ 9] Fwd Packet Length Std
  [10] Bwd Packet Length Max
  [11] Bwd Packet Length Min
  [12] Bwd Packet Length Mean
  [13] Bwd Packet Length Std
  [14] Flow Bytes/s
  [15] Flow Packets/s
  [16] Flow IAT Mean
  [17] Flow IAT Std
  [18] Flow IAT Max
  [19] Flow IAT Min
  [20] Fwd IAT Total
  [21] Fwd IAT Mean
  [22] Fwd IAT Std
  [23] Fwd IAT Max
  [24] Fwd IAT Min
  [25] Bwd IAT Total
  [26] Bwd IAT Mean
  [27] Bwd IAT Std
  [28] Bwd IAT Max
  [29] Bwd IAT Min
  [30] Fwd PSH Flags
  [31] Bwd PSH Flags
  [32] Fwd URG Flags
  [33] Bwd URG Flags
  [34] Fwd H

In [4]:
# Normalize label text: the raw CSVs encode the web-attack dash as a non-UTF-8 byte that
# reads back as the replacement char (e.g. "Web Attack � XSS"). Map it to a plain hyphen
# and strip whitespace so per-class reporting (Flaw 3) uses clean, stable class names.
df["Label"] = df["Label"].str.replace("�", "-", regex=False).str.replace("  ", " ").str.strip()

# Label distribution — counts and percentages
label_counts = df["Label"].value_counts()
label_pct = df["Label"].value_counts(normalize=True) * 100
label_summary = pd.DataFrame({"count": label_counts, "pct": label_pct.round(4)})
print(label_summary.to_string())

benign_pct = label_pct.get("BENIGN", 0.0)
print(f"\nBENIGN share: {benign_pct:.1f}%   |   Attack share: {100 - benign_pct:.1f}%")
print(f"Distinct labels: {df['Label'].nunique()}")

                              count      pct
Label                                       
BENIGN                      2273097  80.3004
DoS Hulk                     231073   8.1630
PortScan                     158930   5.6144
DDoS                         128027   4.5227
DoS GoldenEye                 10293   0.3636
FTP-Patator                    7938   0.2804
SSH-Patator                    5897   0.2083
DoS slowloris                  5796   0.2048
DoS Slowhttptest               5499   0.1943
Bot                            1966   0.0695
Web Attack - Brute Force       1507   0.0532
Web Attack - XSS                652   0.0230
Infiltration                     36   0.0013
Web Attack - Sql Injection       21   0.0007
Heartbleed                       11   0.0004

BENIGN share: 80.3%   |   Attack share: 19.7%
Distinct labels: 15


### 1.3 — Quality issue: Inf and NaN values

CICFlowMeter divides by flow duration to produce `Flow Bytes/s` and `Flow Packets/s`; zero-duration
flows yield `Inf` (Engelen et al., 2021). We count these, convert `±Inf` to `NaN`, report which
columns carry missing values, and drop the affected rows. This minimal cleaning is identical for
both evaluation strategies, so doing it once here introduces no leakage.

In [5]:
numeric_cols = df.select_dtypes(include=np.number).columns

# 1. Count Inf cells, then convert to NaN
inf_mask = np.isinf(df[numeric_cols])
inf_total = int(inf_mask.to_numpy().sum())
inf_by_col = inf_mask.sum()
inf_by_col = inf_by_col[inf_by_col > 0]
print(f"Total Inf/-Inf cells: {inf_total:,}")
print("Columns carrying Inf:")
print(inf_by_col.to_string())

df = df.replace([np.inf, -np.inf], np.nan)

# 2. NaN counts per column (after Inf conversion)
nan_counts = df.isna().sum()
nan_cols = nan_counts[nan_counts > 0]
print(f"\nColumns with NaN (post Inf->NaN): {len(nan_cols)}")
print(nan_cols.to_string())

# 3. Drop rows with any NaN (document the count)
n_before = len(df)
df = df.dropna().reset_index(drop=True)
print(f"\nDropped {n_before - len(df):,} rows containing NaN ({(n_before - len(df)) / n_before * 100:.3f}%)")
print(f"Remaining rows: {len(df):,}")

Total Inf/-Inf cells: 4,376
Columns carrying Inf:
Flow Bytes/s      1509
Flow Packets/s    2867



Columns with NaN (post Inf->NaN): 2
Flow Bytes/s      2867
Flow Packets/s    2867



Dropped 2,867 rows containing NaN (0.101%)
Remaining rows: 2,827,876


In [6]:
# Negative values in flow features that are physically non-negative (counts, lengths, durations).
# CICFlowMeter is known to emit spurious negatives (Engelen et al., 2021); we document, not fix, here.
feature_cols = [c for c in numeric_cols if c not in ("day",)]
neg_counts = (df[feature_cols] < 0).sum()
neg_cols = neg_counts[neg_counts > 0].sort_values(ascending=False)
print(f"Feature columns containing negative values: {len(neg_cols)}")
print(neg_cols.to_string() if len(neg_cols) else "  (none)")

Feature columns containing negative values: 13
Init_Win_bytes_backward    1439672
Init_Win_bytes_forward     1001172
Flow IAT Min                  2890
Flow Packets/s                 115
Flow Duration                  115
Flow IAT Mean                  115
Flow IAT Max                   115
Flow Bytes/s                    85
Fwd Header Length               35
Fwd Header Length.1             35
min_seg_size_forward            35
Bwd Header Length               22
Fwd IAT Min                     17


### 1.4 — Quality issue: duplicate rows (Flaw 1)

CICIDS2017 contains a large number of **exact duplicate flow records** (Engelen et al., 2021). Under
a random train/test split these duplicates are scattered across both sets, letting the model
"memorize" test points it has already seen in training — inflating every metric. This is **Flaw 1**
of our critique.

We **quantify** the duplicates here and persist the count to `figures/dedup_stats.txt`, but we do
**not** remove them yet. Deduplication is deferred to **Strategy B** so we can measure the exact
F1 inflation that the duplicates cause (run A0 with duplicates vs A1 without).

In [7]:
# Duplicates over the feature + label columns (ignore the tag columns we added).
dedup_subset = [c for c in df.columns if c not in ("day", "day_name")]
n_dupes = int(df.duplicated(subset=dedup_subset).sum())
pct_dupes = n_dupes / len(df) * 100
print(f"FLAW 1 - exact duplicate rows: {n_dupes:,} ({pct_dupes:.2f}% of data)")
print("Duplicates are RETAINED here on purpose; removal is deferred to Strategy B (A0 vs A1).")

# Persist for the report (cited verbatim in the critique)
stats_path = FIGURES_DIR / "dedup_stats.txt"
stats_path.write_text(
    f"rows_total={len(df)}\n"
    f"duplicate_rows={n_dupes}\n"
    f"duplicate_pct={pct_dupes:.4f}\n",
    encoding="utf-8",
)
print(f"Saved: {stats_path}")

FLAW 1 - exact duplicate rows: 307,078 (10.86% of data)
Duplicates are RETAINED here on purpose; removal is deferred to Strategy B (A0 vs A1).
Saved: ..\figures\dedup_stats.txt


### 1.5 — Zero-variance columns, artifacts, and memory

We list **zero-variance** feature columns (constant for every row) — they carry no signal and are
dropped in Phase 3 — count the `Flow Duration == 0` CICFlowMeter artifacts, and report the
DataFrame's memory footprint to justify subsampling during development (ADR-10).

In [8]:
# Zero-variance feature columns (constant) — candidates to drop in Phase 3
feature_only = df.drop(columns=["Label", "day", "day_name"]).select_dtypes(include=np.number)
zero_var_cols = feature_only.columns[feature_only.nunique() <= 1].tolist()
print(f"Zero-variance (constant) feature columns: {len(zero_var_cols)}")
for c in zero_var_cols:
    print(f"  - {c}")

# Flow Duration == 0 artifacts
if "Flow Duration" in df.columns:
    n_zero_dur = int((df["Flow Duration"] == 0).sum())
    print(f"\nFlows with Flow Duration == 0 (CICFlowMeter artifact): {n_zero_dur:,}")

# Memory footprint
mem_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"\nDataFrame memory footprint: {mem_mb:,.1f} MB")

Zero-variance (constant) feature columns: 8
  - Bwd PSH Flags
  - Bwd URG Flags
  - Fwd Avg Bytes/Bulk
  - Fwd Avg Packets/Bulk
  - Fwd Avg Bulk Rate
  - Bwd Avg Bytes/Bulk
  - Bwd Avg Packets/Bulk
  - Bwd Avg Bulk Rate

Flows with Flow Duration == 0 (CICFlowMeter artifact): 0



DataFrame memory footprint: 2,005.1 MB


### 1.6 — Data-quality summary

**Findings (all consistent with Engelen et al., 2021):**

- Column headers carried whitespace → stripped on load.
- `Flow Bytes/s` and `Flow Packets/s` carried `Inf` from divide-by-zero → converted to `NaN`; the
  affected rows were dropped (identical minimal cleaning for both strategies, so no leakage).
- A substantial fraction of rows are **exact duplicates** — the core leakage vector (Flaw 1). These
  are deliberately **kept** for now; the count is saved to `figures/dedup_stats.txt`.
- The label distribution is severely imbalanced (~80% BENIGN) with ultra-rare attack classes
  (Heartbleed, SQL Injection, Infiltration), foreshadowing Flaw 3.
- Zero-variance columns and the redundant `Fwd Header Length.1` are flagged for removal in Phase 3.

**Deferral note (deliberate):** Deduplication is *not* applied here. It is applied only inside
**Strategy B** so the difference between the duplicate-contaminated random split (A0) and the
deduplicated random split (A1) directly measures the F1 inflation caused by Flaw 1. This mirrors
the source paper's pipeline, which never mentions deduplication.